# 📋 Hướng dẫn cho thành viên: đo benchmark M1 trên Range của mình

**Yêu cầu của nhóm trưởng (Bảo).** Mỗi bạn chạy notebook này **một lượt chính thức** trên
**cả Range của mình**, bằng **tài khoản Colab của mình**, ở **Colab CPU**, đúng tag
`m1-bench-v2`. Số liệu của 5 lượt này là số chính thức cho báo cáo M1 (trang *Kết quả thu
thập* và *Hiệu năng*). Số tự báo trước đây chỉ để đối chiếu.

**Hạn: chạy xong và chia sẻ kết quả trước hết ngày 17/11/2026.** Một Range cần khoảng
9–12 giờ, tức 1–2 phiên Colab. Nên **chạy thử trước 12/11** và **bắt đầu lượt chính thức
chậm nhất 13/11**, để còn thời gian chạy lại nếu lỗi.

| Bạn | `MEMBER` | Range | Version (09/11) |
|---|---|---|---|
| Bảo | `bao` | `2508.02872`–`2508.07871` | 7.835 |
| Minh | `minh` | `2409.18594`–`2409.20569` + `2410.00001`–`2410.03024` | 8.503 |
| Nghiêm | `nghiem` | `2409.08594`–`2409.13593` | 8.090 |
| Hải | `hai` | `2412.08848`–`2412.13847` | 8.124 |
| Liễu | `lieu` | `2410.08025`–`2410.13024` | 8.408 |

### Bước 0. Chuẩn bị (một lần)
1. Mở notebook từ link nhóm trưởng gửi (đường dẫn có `blob/m1-bench-v2/`).
   *File → Save a copy in Drive* nếu muốn giữ bản của mình.
2. *Runtime → Change runtime type → **CPU***. Không dùng GPU/TPU: đề đo trên Colab CPU.
3. Mở 🔑 **Secrets** (thanh bên trái), thêm và bật *Notebook access* cho:
   - `CONTACT_EMAIL`: email của bạn (arXiv yêu cầu User-Agent có email liên hệ);
   - `S2_API_KEY`: key Semantic Scholar **của chính bạn**.
   - `HF_TOKEN`: token HF **của chính bạn**, quyền Write trên org `hcmus-arxiv-reference` (trừ Bảo);
   - `HF_REPO_ID`: `hcmus-arxiv-reference/arXiv-reference-matching`.

   **Không dán key, token hay email vào ô code.**

### Bước 1. Chạy thử (khoảng 10 phút, không nộp)
Trong ô **Tham số**: `MEMBER` = tên bạn, `ID_MODE = "member"`, `N_IDS = 30`. Bấm
*Runtime → Run all*. Đạt khi chạy hết không lỗi và ô **Lưu** (gần cuối) in **`TRẠNG THÁI: XONG`**.
Lỗi thì xem mục *Gặp lỗi* ở cuối notebook, rồi báo nhóm kèm ảnh chụp lỗi và file log.

### Bước 2. Lượt chính thức
1. *Runtime → Disconnect and delete runtime* (bắt đầu từ runtime mới).
2. Ô **Tham số**: `MEMBER` = tên bạn, `ID_MODE = "range"`. Giữ nguyên mọi thứ khác.
3. *Run all*. **Giữ tab trình duyệt mở** và máy không ngủ. Mỗi 45 phút notebook nén một
   checkpoint lên Drive (`MyDrive/arxiv-m1-benchmark/`) và in dòng `run_NNN: ...`.
4. Notebook tự dừng vòng crawl trước 11 giờ. Nếu ô **Lưu** in **`CHƯA XONG`** (hoặc Colab
   ngắt giữa chừng): mở runtime mới, đặt `RESTORE_ARCHIVE` bằng đường dẫn checkpoint mà
   notebook in ra (`.../<RUN_ID>__checkpoint.tar.gz`), **giữ nguyên `REF`**, *Run all* lại.
   Lặp tới khi in **`TRẠNG THÁI: XONG`**.

### Bước 3. Nộp kết quả
Xem ô **Nộp kết quả** ở cuối notebook.

### Không được làm
- Không đổi `REF`, không sửa ô code nào ngoài ô **Tham số**: cả nhóm phải đo cùng một code.
- Không chạy hai lượt cùng lúc (hai tab Colab, hoặc Colab cùng lúc với crawler trên máy):
  arXiv yêu cầu một kết nối, cách nhau ≥ 3 giây; chạy song song sẽ bị chặn và làm sai số đo.
- Upload lên Hugging Face bật sẵn (`UPLOAD_TO_HF`, trừ Range Bảo). Trước khi chạy: chấp nhận lời mời vào org
  `hcmus-arxiv-reference` trên HF, tạo token fine-grained của **chính bạn** có quyền Write trên org đó,
  rồi thêm vào Secrets `HF_TOKEN` và `HF_REPO_ID` = `hcmus-arxiv-reference/arXiv-reference-matching`.
  Upload chạy ở cuối, chỉ khi crawl xong và validator đạt; file lên `raw/<MEMBER>/`, bảng lên `tables/<MEMBER>/`.
- Không xóa file checkpoint trên Drive cho tới khi báo cáo M1 nộp xong.

# Benchmark M1 trên Google Colab (CPU): cả Range hoặc tự kiểm thử

Notebook chạy Engine `src/crawl/` (`python -m src.crawl.cli`, `CrawlPipeline`, ADR 0001) trong testbed
của đề (Colab CPU-only) và lưu mọi thứ cần cho báo cáo M1: dữ liệu, checkpoint, số liệu thời
gian/RAM/disk, thống kê và validator.

1. Clone repo tại `REF` (nhánh, tag hoặc SHA), ghi commit thực tế vào `run.json`.
2. Cài `requirements.txt`, chạy `pytest` (log ra file, lỗi thì dừng).
3. Chọn ID: `range` = cả Range của `MEMBER` trong `configs/ranges.toml` (kể cả Range nhiều Đoạn);
   `member` = `N_IDS` ID quanh ranh giới Đoạn; `explicit` = danh sách cụ thể.
4. Crawl qua harness `src.benchmark` vào `/content/`, **chia thành các đoạn** dài tối đa
   `SEGMENT_MINUTES`. Hết giờ, notebook đặt `STOP_RANGE`; Runner dừng sau Publication đang chạy
   và xuất bảng. Giữa hai đoạn, notebook nén cả lượt chạy thành **một** `.tar.gz` và thay
   checkpoint trên Drive (bản trước giữ ở `.prev`). Crawler không chạy lúc nén nên bản nén nhất
   quán. Mỗi đoạn là một `bench/run_NNN`; `src.merge_benchmarks` cộng wall time các đoạn, lấy RAM
   đỉnh/TB và disk đỉnh/cuối. Thời gian nén checkpoint không tính vào wall time.
5. Khi crawl xong: chạy lại với **mạng bị chặn** trên các Publication hoàn chỉnh (phải không cần
   mạng, không đổi task, không đổi checksum).
6. `make_stats` (gồm thời gian mỗi bài và tổng thời gian Discovery từ `manifests/timings.jsonl`),
   gộp benchmark, validator, số lần HTTP 429 và thời gian chờ backoff.
7. **Bộ số liệu cho báo cáo** `report/` (cùng layout với `data/m1/` của repo báo cáo):
   `snapshots/<MEMBER>.json` (`scripts.report_m1`: Publication/Version, kích thước trước/sau
   khi bỏ hình, Reference, kiểm bảng, hiệu năng) và `benchmark/<MEMBER>/benchmark_total.json`,
   `benchmark_samples_all.csv` (thời gian, Discovery, RAM, disk theo từng đoạn).
8. Checkpoint cuối và các file kết quả lên Drive; tùy chọn upload lên Hugging Face.

**Nhiều phiên:** Colab miễn phí ngắt phiên sau khoảng 12 giờ và khi để rảnh. Notebook tự dừng vòng
crawl trước `SESSION_HOURS`. Phiên mới: mở runtime mới, đặt `RESTORE_ARCHIVE` là đường dẫn
checkpoint trên Drive (notebook in ra sau mỗi đoạn), giữ nguyên `REF`, rồi *Run all*. Notebook kiểm
sha256, giải nén về đúng chỗ cũ trong `/content/` rồi crawl tiếp. Mỗi phiên ghi máy và commit riêng
trong `run.json`. Giữ tab trình duyệt mở trong lúc chạy.

**Bí mật:** mở 🔑 *Secrets* ở thanh bên trái Colab, thêm `CONTACT_EMAIL` (bắt buộc, arXiv yêu cầu
User-Agent có email), `S2_API_KEY` (key của chính bạn), và chỉ khi upload: `HF_TOKEN`,
`HF_REPO_ID`. Bật *Notebook access*. Không dán token hay key vào ô code.

**Lưu trữ:** crawl vào `/content/` để đo hiệu năng local. `/content/` mất khi runtime kết thúc;
Drive chỉ giữ file nén. Không crawl thẳng vào Drive: đọc/ghi nhiều file nhỏ trên Drive chậm và có
thể chạm giới hạn I/O ([Colab FAQ](https://research.google.com/colaboratory/faq.html)). Mỗi lượt có
`RUN_ID` riêng; thư mục đã tồn tại thì notebook dừng, không ghi đè. Không có nút xóa dữ liệu.

**Lịch sự với arXiv:** mỗi người chỉ chạy **một** lượt tại một thời điểm, trên Range của mình
(một kết nối, ≥ 3 giây giữa các request; Engine tự giữ nhịp).

## ✏️ Tham số: ô duy nhất bạn cần sửa

Thường chỉ đổi `MEMBER`, `ID_MODE` (và `N_IDS` khi chạy thử), và `RESTORE_ARCHIVE` khi chạy
phiên tiếp theo. Các dòng còn lại giữ nguyên.

In [ ]:
# ==== Tham số ====
REPO_URL = "https://github.com/hcmus-arxiv-reference/arxiv-reference-matching.git"
REF = "m1-bench-v2"     # tag benchmark của nhóm: cả 5 người cùng REF, KHÔNG đổi

ID_MODE = "range"       # "range": cả Range của MEMBER; "member": N_IDS ID; "explicit": EXPLICIT_IDS
MEMBER = "bao"          # tên trong configs/ranges.toml: bao, minh, nghiem, hai, lieu
N_IDS = 20              # chỉ dùng với "member": lấy quanh ranh giới Đoạn đầu tiên
EXPLICIT_IDS = ["2409.20569", "2410.00001", "2410.08025"]

SEGMENT_MINUTES = 45    # độ dài tối đa một đoạn crawl; sau mỗi đoạn có một checkpoint
SESSION_HOURS = 11      # dừng vòng crawl trước khi Colab ngắt phiên (~12 giờ)
MAX_SEGMENTS = None     # giới hạn số đoạn trong phiên này (None = không giới hạn); để chạy thử
CRAWL_ARGS = ["--max-passes", "3", "--retry-delay", "600"]

RUN_PYTEST = True
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/arxiv-m1-benchmark"
RESTORE_ARCHIVE = None  # phiên mới: f"{DRIVE_DIR}/<RUN_ID>__checkpoint.tar.gz"
ALLOW_COMMIT_CHANGE = False   # True chỉ khi cố ý đổi commit giữa các phiên của một lượt

UPLOAD_TO_HF = MEMBER != "bao"   # Range Bảo giữ bản crawl trên Mac ở raw/bao/; các Range khác upload khi xong

WORK_ROOT = "/content/m1-benchmark"
REPO_DIR = "/content/arxiv-reference-matching"
BLOCKED_PROXY = "http://127.0.0.1:9"   # cổng discard: mọi request HTTP(S) đều thất bại

In [ ]:
# Hàm dùng chung. Mọi lệnh ghi stdout/stderr ra file log để tải về khi lỗi.
import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, time
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SESSION = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
SESSION_STARTED = time.monotonic()
PENDING_LOGS = Path(WORK_ROOT, "sessions", SESSION)   # log trước khi có RUN_DIR
PENDING_LOGS.mkdir(parents=True, exist_ok=True)


def run(cmd, log, *, check=True, env=None, cwd=None):
    """Chạy lệnh, ghi log; check=True thì mã thoát khác 0 sẽ ném lỗi kèm 20 dòng cuối."""
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    with log.open("w", encoding="utf-8") as stream:
        stream.write("$ " + " ".join(map(str, cmd)) + "\n")
        stream.flush()
        result = subprocess.run(list(map(str, cmd)), stdout=stream, stderr=subprocess.STDOUT,
                                env=env, cwd=cwd, text=True)
    seconds = round(time.monotonic() - started, 2)
    print(f"exit={result.returncode} {seconds}s log={log}")
    if check and result.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {result.returncode}): {cmd}\n{tail(log)}")
    return {"cmd": list(map(str, cmd)), "returncode": result.returncode,
            "seconds": seconds, "log": str(log)}


def tail(log, n=20):
    return "\n".join(Path(log).read_text(encoding="utf-8").splitlines()[-n:])


def sha256_file(path):
    with open(path, "rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def data_checksum(data_dir):
    """sha256 trên danh sách file dữ liệu đã sắp xếp: raw/, metadata/, references/.

    Bỏ manifests/ và tables/: checkpoint và bảng xuất lại được ghi lại hợp lệ mỗi lượt.
    """
    data_dir = Path(data_dir)
    files = sorted(p for d in ("raw", "metadata", "references") for p in (data_dir / d).rglob("*")
                   if p.is_file())
    digest = hashlib.sha256()
    for path in files:
        digest.update(f"{path.relative_to(data_dir).as_posix()}\0{sha256_file(path)}\n".encode())
    return {"files": len(files), "sha256": digest.hexdigest()}


def task_state(data_dir, ids=None):
    """Bản ghi cuối của mỗi task trong crawl.jsonl: (status, attempts, checksum)."""
    state = {}
    for line in Path(data_dir, "manifests", "crawl.jsonl").read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        if ids is None or r["arxiv_id"] in ids:
            state[f"{r['arxiv_id']}|{r['version']}|{r['kind']}"] = [r["status"], r["attempts"], r["checksum"]]
    return state


def http_retries(data_dir):
    """429/5xx và thời gian chờ từ manifests/http-events.jsonl (chỉ tải source arXiv)."""
    path = Path(data_dir, "manifests", "http-events.jsonl")
    events = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()] if path.exists() else []
    retries = [e for e in events if e["event"] == "retry"]
    return {"http_429": sum(e.get("status") == 429 for e in retries),
            "retries": len(retries),
            "backoff_seconds": round(sum(e.get("delay", 0) for e in retries), 1),
            "partial": sum(e["event"] == "partial" for e in events),
            "resume": sum(e["event"] == "resume" for e in events),
            "scope": "source downloads only; discovery/S2 retries are not logged here"}

In [ ]:
# Clone repo tại REF và ghi commit thực tế (SHA đầy đủ).
if not os.path.isdir(REPO_DIR):
    run(["git", "clone", REPO_URL, REPO_DIR], PENDING_LOGS / "git-clone.log")
run(["git", "-C", REPO_DIR, "fetch", "origin", REF], PENDING_LOGS / "git-fetch.log")
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"], PENDING_LOGS / "git-checkout.log")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("ref:", REF, "commit:", COMMIT)
# REF quá cũ thì thiếu phép đo thời gian Discovery/mỗi bài (manifests/timings.jsonl): dừng sớm.
assert "timings.jsonl" in Path("src/crawl/pipeline.py").read_text(encoding="utf-8"), \
    "REF chưa có phép đo thời gian Discovery; dùng commit đã gộp PR #34"
assert "--ids-file" in Path("scripts/report_m1.py").read_text(encoding="utf-8"), \
    "REF chưa có report_m1 cho mọi Range; dùng tag benchmark mới nhất (m1-bench-v2 trở đi)"

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], PENDING_LOGS / "pip.log")

In [ ]:
# Bí mật: Colab Secrets (hoặc biến môi trường khi chạy ngoài Colab). Không in giá trị.
def secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

NAMES = ["CONTACT_EMAIL", "S2_API_KEY"] + (["HF_TOKEN", "HF_REPO_ID"] if UPLOAD_TO_HF else [])
for name in NAMES:
    value = secret(name)
    if value:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)
    print(f"{name}: {'đã đặt' if value else 'chưa đặt'}")
assert os.environ.get("CONTACT_EMAIL"), "Thêm CONTACT_EMAIL trong Colab Secrets trước khi crawl"
if UPLOAD_TO_HF:
    assert os.environ.get("HF_TOKEN") and os.environ.get("HF_REPO_ID"), "Upload cần HF_TOKEN và HF_REPO_ID"
for name in ("HTTPS_PROXY", "HTTP_PROXY", "https_proxy", "http_proxy"):
    os.environ.pop(name, None)   # crawl cần mạng thật

In [ ]:
# Thư mục lượt chạy. Lượt mới: RUN_ID mới, đã tồn tại thì dừng. Phiên mới: khôi phục checkpoint.
import psutil

MACHINE = {"platform": platform.platform(), "python": sys.version, "cpus": os.cpu_count(),
           "ram_gib": round(psutil.virtual_memory().total / 2**30, 1),
           "disk_free_gib": round(shutil.disk_usage(WORK_ROOT).free / 2**30), "colab": IN_COLAB}
if SAVE_TO_DRIVE or RESTORE_ARCHIVE:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

RUNS = Path(WORK_ROOT, "runs")
if RESTORE_ARCHIVE:
    archive = Path(RESTORE_ARCHIVE)
    expected = Path(f"{archive}.sha256").read_text().split()[0]
    actual = sha256_file(archive)
    assert actual == expected, f"sha256 không khớp: {actual} != {expected}"
    with tarfile.open(archive) as tar:
        top = {Path(m.name).parts[0] for m in tar.getmembers()}
        assert len(top) == 1, f"Bản lưu phải có đúng một thư mục gốc, có {top}"
        RUN_ID = top.pop()
        RUN_DIR = RUNS / RUN_ID
        if RUN_DIR.exists():
            raise FileExistsError(f"{RUN_DIR} đã tồn tại; dùng runtime mới hoặc đổi WORK_ROOT")
        RUNS.mkdir(parents=True, exist_ok=True)
        tar.extractall(RUNS, filter="data")
    RUN = json.loads((RUN_DIR / "run.json").read_text(encoding="utf-8"))
    print("Khôi phục", RUN_ID, "từ", archive, "sha256 OK")
else:
    RUN_ID = f"{SESSION}_{COMMIT[:7]}"
    RUN_DIR = RUNS / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)   # không bao giờ ghi đè lượt cũ
    RUN = {"run_id": RUN_ID, "repo": REPO_URL, "sessions": [], "segments": [], "checkpoints": []}
DATA_DIR = RUN_DIR / "data"
BENCH_DIR = RUN_DIR / "bench"
SESSION_DIR = RUN_DIR / "sessions" / SESSION
SESSION_DIR.mkdir(parents=True, exist_ok=False)
for log in PENDING_LOGS.iterdir():
    shutil.move(str(log), SESSION_DIR / log.name)
THIS = {"session": SESSION, "ref": REF, "commit": COMMIT, "machine": MACHINE,
        "restored_from": str(RESTORE_ARCHIVE) if RESTORE_ARCHIVE else None,
        "s2_api_key": bool(os.environ.get("S2_API_KEY")), "steps": {}}
RUN["sessions"].append(THIS)
FIRST_COMMIT = RUN["sessions"][0]["commit"]
if FIRST_COMMIT != COMMIT:
    message = f"commit phiên này {COMMIT[:7]} khác commit phiên đầu {FIRST_COMMIT[:7]}"
    assert ALLOW_COMMIT_CHANGE, message + "; đặt REF đúng SHA cũ (hoặc ALLOW_COMMIT_CHANGE=True)"
    print("CHÚ Ý:", message)


def save_run():
    (RUN_DIR / "run.json").write_text(json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")


def checkpoint():
    """Nén RUN_DIR thành một .tar.gz, thay checkpoint trên Drive (bản trước thành .prev).

    Chỉ gọi khi crawler không chạy. Ghi ra .part rồi đổi tên, nên checkpoint cũ không bao
    giờ bị thay bằng một bản chép dở.
    """
    started = time.monotonic()
    save_run()
    archives = Path(WORK_ROOT, "archives")
    archives.mkdir(parents=True, exist_ok=True)
    name = f"{RUN_ID}__checkpoint.tar.gz"
    part = archives / f"{name}.part"
    with tarfile.open(part, "w:gz", compresslevel=1) as tar:
        tar.add(RUN_DIR, arcname=RUN_ID)
    digest = sha256_file(part)
    item = {"session": SESSION, "at": datetime.now(timezone.utc).isoformat(),
            "segments": len(RUN["segments"]), "bytes": part.stat().st_size, "sha256": digest}
    if SAVE_TO_DRIVE:
        target = Path(DRIVE_DIR, name)
        remote_part = Path(DRIVE_DIR, f"{name}.part")
        shutil.copyfile(part, remote_part)
        assert sha256_file(remote_part) == digest, "bản chép lên Drive sai sha256"
        if target.exists():
            os.replace(target, Path(DRIVE_DIR, f"{name}.prev"))
            os.replace(Path(f"{target}.sha256"), Path(DRIVE_DIR, f"{name}.prev.sha256"))
        os.replace(remote_part, target)
        Path(f"{target}.sha256").write_text(f"{digest}  {name}\n")
        item["drive"] = str(target)
    else:
        os.replace(part, archives / name)
        Path(archives, f"{name}.sha256").write_text(f"{digest}  {name}\n")
        item["local"] = str(archives / name)
    part.unlink(missing_ok=True)
    item["seconds"] = round(time.monotonic() - started, 1)
    RUN["checkpoints"].append(item)
    save_run()
    print(f"checkpoint {item['bytes'] / 2**20:.1f} MiB trong {item['seconds']}s:",
          item.get("drive") or item["local"])
    return item


save_run()
print("RUN_ID:", RUN_ID)
print(json.dumps(MACHINE, indent=2))

In [ ]:
if RUN_PYTEST:
    env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    THIS["steps"]["pytest"] = run([sys.executable, "-m", "pytest", "-q"], SESSION_DIR / "pytest.log",
                                  check=False, env=env)
    save_run()
    lines = Path(THIS["steps"]["pytest"]["log"]).read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[-30:] if THIS["steps"]["pytest"]["returncode"] else lines[-1:]))
    assert THIS["steps"]["pytest"]["returncode"] == 0, "pytest thất bại; xem log (kiểm phiên bản Python trước)"

In [ ]:
# Chọn ID. Lượt khôi phục dùng lại đúng danh sách đã lưu.
from src.common.config import load_ranges, member_ids


def around_boundary(member, n):
    """n ID liên tiếp quanh ranh giới Đoạn đầu tiên; Range một Đoạn thì lấy n ID đầu."""
    segments = load_ranges()[member].range.segments
    ids = [i for s in segments for i in s.ids()]
    if len(segments) == 1:
        return ids[:n]
    boundary = len(segments[0])
    start = max(0, boundary - n // 2)
    picked = ids[start:start + n]
    assert picked[0] in segments[0] and picked[-1] in segments[1], "chưa phủ hai bên ranh giới"
    return picked


if RESTORE_ARCHIVE:
    IDS = RUN["ids"]
else:
    if ID_MODE == "range":
        IDS, mode = list(member_ids(MEMBER)), f"range:{MEMBER}"
    elif ID_MODE == "member":
        IDS, mode = around_boundary(MEMBER, N_IDS), f"member:{MEMBER}:{N_IDS}"
    elif ID_MODE == "explicit":
        IDS, mode = list(dict.fromkeys(EXPLICIT_IDS)), "explicit"
    else:
        raise ValueError(f"ID_MODE không hợp lệ: {ID_MODE}")
    RUN.update(id_mode=mode, member=MEMBER if ID_MODE != "explicit" else None, ids=IDS)
IDS_FILE = RUN_DIR / "ids.txt"
if not IDS_FILE.exists():
    IDS_FILE.write_text("\n".join(IDS) + "\n", encoding="utf-8")
save_run()
print(RUN["id_mode"], len(IDS), "ID:", IDS[0], "…", IDS[-1])

## Crawl theo đoạn, checkpoint giữa các đoạn

Vòng lặp dừng khi Runner báo `complete` (mọi Publication hoàn chỉnh) hoặc `incomplete` (đã hết
`--max-passes` lượt thử lại trong một đoạn; thường là các Version HTTP 404), khi gần hết
`SESSION_HOURS`, hoặc khi bạn bấm dừng ô (notebook đặt `STOP_RANGE`, chờ crawler dừng gọn rồi
checkpoint). Chạy lại ô này trong cùng phiên để crawl tiếp.

In [ ]:
# Crawl chạy qua harness đo RAM/disk. Rẽ nhánh theo cả mã thoát lẫn status của Runner.
STOP_FILE = DATA_DIR / "STOP_RANGE"


def crawl_segment(out, log, ids_file, env, budget=None):
    """Một đoạn benchmark; quá `budget` giây thì đặt STOP_RANGE để Runner dừng gọn."""
    cmd = [sys.executable, "-m", "src.benchmark", "--data-dir", DATA_DIR, "--out", out, "--",
           sys.executable, "-m", "src.crawl.cli", "--ids-file", ids_file,
           "--data-dir", DATA_DIR, *CRAWL_ARGS]
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    STOP_FILE.unlink(missing_ok=True)
    log.parent.mkdir(parents=True, exist_ok=True)
    started, reason = time.monotonic(), None
    with log.open("w", encoding="utf-8") as stream:
        stream.write("$ " + " ".join(map(str, cmd)) + "\n")
        stream.flush()
        proc = subprocess.Popen(list(map(str, cmd)), stdout=stream, stderr=subprocess.STDOUT,
                                env=env, text=True)
        try:
            while proc.poll() is None:
                if budget is not None and reason is None and time.monotonic() - started > budget:
                    STOP_FILE.touch()
                    reason = "segment_budget"
                time.sleep(5)
        except KeyboardInterrupt:
            STOP_FILE.touch()      # dừng gọn: Runner xong Publication hiện tại rồi xuất bảng
            reason = "interrupted"
            proc.wait()
    STOP_FILE.unlink(missing_ok=True)
    report = json.loads((DATA_DIR / "manifests" / "range-run.json").read_text(encoding="utf-8"))
    step = {"session": SESSION, "out": str(out.relative_to(RUN_DIR)), "log": str(log.relative_to(RUN_DIR)),
            "returncode": proc.returncode, "seconds": round(time.monotonic() - started, 2),
            "stop_reason": reason, "engine": report.get("engine"), "status": report["status"],
            "pass": report["pass"],
            "visited_this_pass": report["visited_this_pass"], "counts": report["counts"],
            "pending": len(report["pending_ids"]),
            "benchmark": json.loads((out / "benchmark.json").read_text())}
    return step, report


def crawl_until_done():
    """Chạy các đoạn tới khi Runner xong, hết giờ phiên hoặc bị dừng tay."""
    deadline = SESSION_STARTED + SESSION_HOURS * 3600
    done_this_session = 0
    while not RUN.get("crawl_finished"):
        remaining = deadline - time.monotonic()
        if remaining < 15 * 60:
            print("Gần hết SESSION_HOURS: dừng. Phiên mới dùng RESTORE_ARCHIVE ở dòng checkpoint trên.")
            return
        if MAX_SEGMENTS is not None and done_this_session >= MAX_SEGMENTS:
            print(f"Đã chạy MAX_SEGMENTS={MAX_SEGMENTS} đoạn trong phiên này: dừng.")
            return
        label = f"run_{len(RUN['segments']) + 1:03d}"
        step, report = crawl_segment(BENCH_DIR / label, SESSION_DIR / f"crawl-{label}.log", IDS_FILE,
                                     dict(os.environ), budget=min(SEGMENT_MINUTES * 60, remaining - 10 * 60))
        step["segment"] = label
        RUN["segments"].append(step)
        assert step["engine"] == "main", f"Engine phải là CrawlPipeline (main), nhận {step['engine']!r}"
        done_this_session += 1
        RUN["crawl_finished"] = step["status"] in ("complete", "incomplete")
        RUN["crawl_complete"] = step["status"] == "complete" and step["returncode"] == 0
        RUN["pending_ids"] = report["pending_ids"]
        print(f"{label}: status={step['status']} exit={step['returncode']} {step['seconds']}s "
              f"complete={step['counts'].get('complete', 0)}/{len(IDS)} pending={step['pending']} "
              f"stop={step['stop_reason']}")
        checkpoint()
        if step["stop_reason"] == "interrupted":
            raise KeyboardInterrupt("Đã dừng gọn và checkpoint; chạy lại ô này để crawl tiếp")
        if step["status"] == "failed" or (step["status"] == "stopped" and step["stop_reason"] is None):
            raise RuntimeError(f"Crawler dừng bất thường ({step['status']}):\n{tail(RUN_DIR / step['log'])}")


crawl_until_done()

In [ ]:
# Chạy lại với mạng bị chặn trên các Publication hoàn chỉnh: không request, không đổi task/checksum.
def resume_check(label):
    ready = [i for i in IDS if i not in set(RUN["pending_ids"])]
    # Lượt kiểm chỉ chạy trên Publication hoàn chỉnh; giữ range-run.json của lượt crawl thật
    # (báo cáo đọc engine, status, thời điểm xong từ file này).
    run_state = DATA_DIR / "manifests" / "range-run.json"
    crawl_state = run_state.read_bytes()
    ids_file = SESSION_DIR / "ids-complete.txt"
    ids_file.write_text("\n".join(ready) + "\n", encoding="utf-8")
    before = {"data": data_checksum(DATA_DIR), "tasks": task_state(DATA_DIR, set(ready))}
    env = dict(os.environ, HTTPS_PROXY=BLOCKED_PROXY, HTTP_PROXY=BLOCKED_PROXY,
               https_proxy=BLOCKED_PROXY, http_proxy=BLOCKED_PROXY)
    env["NO_PROXY"] = env["no_proxy"] = ""  # không cho host bỏ qua proxy chặn mạng
    step, _ = crawl_segment(SESSION_DIR / f"bench-{label}", SESSION_DIR / f"crawl-{label}.log",
                            ids_file, env)
    after = {"data": data_checksum(DATA_DIR), "tasks": task_state(DATA_DIR, set(ready))}
    run_state.write_bytes(crawl_state)
    changed = sorted(k for k in before["tasks"].keys() | after["tasks"].keys()
                     if before["tasks"].get(k) != after["tasks"].get(k))
    step["checks"] = checks = {
        "exit_0": step["returncode"] == 0,
        "status_complete": step["status"] == "complete",
        "visited_0": step["visited_this_pass"] == 0,
        "tasks_unchanged": not changed,
        "data_checksum_unchanged": before["data"] == after["data"],
    }
    step.update(publications=len(ready), data_checksum=after["data"], changed_tasks=changed[:20])
    THIS["steps"][label] = step
    save_run()
    print(len(ready), "Publication;", json.dumps(checks, indent=2))
    assert all(checks.values()), f"Resume không đạt: {checks}; task đổi: {changed[:20]}"


if RUN.get("crawl_finished"):
    resume_check("resume-offline")
else:
    print("Crawl chưa xong: bỏ qua kiểm resume. Phiên mới dùng RESTORE_ARCHIVE để crawl tiếp.")

## Thống kê, gộp benchmark, validator và chẩn đoán

Chạy được cả khi crawl chưa xong (số liệu tạm của các đoạn đã chạy).

In [ ]:
from src.crawl.validation import validate_crawl


def segments_for(ids):
    """Gộp ID thành các Đoạn liên tiếp, dùng cho range.segments của validator."""
    segments = []
    for prefix, number in sorted((i.split(".")[0], int(i.split(".")[1])) for i in ids):
        if segments and segments[-1]["prefix"] == prefix and segments[-1]["end"] == number - 1:
            segments[-1]["end"] = number
        else:
            segments.append({"prefix": prefix, "start": number, "end": number})
    return segments


stats_dir = SESSION_DIR / "make_stats"
THIS["steps"]["make_stats"] = run([sys.executable, "-m", "src.make_stats", "--data-dir", DATA_DIR,
                                   "--ids-file", IDS_FILE, "--out", stats_dir],
                                  SESSION_DIR / "make_stats.log")
summary = json.loads((stats_dir / "summary.json").read_text())
merged_dir = SESSION_DIR / "benchmark_total"
THIS["steps"]["merge_benchmarks"] = run([sys.executable, "-m", "src.merge_benchmarks", "--runs", BENCH_DIR,
                                         "--out", merged_dir], SESSION_DIR / "merge_benchmarks.log")
total = json.loads((merged_dir / "benchmark_total.json").read_text())
attempted = len({k.split("|")[0] for k, v in task_state(DATA_DIR).items() if k.endswith("|discovery") and v[1] > 0})
performance = {
    # Thời gian: wall time = tổng các đoạn (không gồm thời gian nén checkpoint giữa các đoạn).
    "total_wall_seconds": total["total_wall_seconds"],
    "avg_wall_seconds_per_requested_paper": round(total["total_wall_seconds"] / len(IDS), 3),
    "avg_seconds_per_paper": summary["avg_seconds_per_paper"],
    "median_seconds_per_paper": summary["median_seconds_per_paper"],
    "max_seconds_per_paper": summary["max_seconds_per_paper"],
    "discovery_seconds_total": summary["discovery_seconds_total"],
    # Bộ nhớ: RSS cả cây tiến trình; disk = thư mục dữ liệu.
    "peak_rss_mb": total["peak_rss_mb"], "avg_rss_mb": total["avg_rss_mb"],
    "peak_disk_mb": total["peak_disk_mb"], "final_disk_mb": total["final_disk_mb"],
    "segments": total["segments"], "sessions": len(RUN["sessions"]),
}
claim = {
    "schema_version": 1, "member": RUN.get("member") or RUN.get("id_mode"), "commit": FIRST_COMMIT,
    "range": {"segments": segments_for(IDS)}, "ids_attempted": attempted,
    "run": {"cold_run": True, "finished": bool(RUN.get("crawl_finished")),
            "machines": [s["machine"]["platform"] for s in RUN["sessions"]],
            "python": RUN["sessions"][0]["machine"]["python"],
            "commits": sorted({s["commit"] for s in RUN["sessions"]}),
            "s2_api_key": all(s["s2_api_key"] for s in RUN["sessions"]),
            "measured_by": "src.benchmark (psutil; RSS cả cây tiến trình, disk qua os.walk), "
                           "gộp bằng src.merge_benchmarks; thời gian mỗi bài từ manifests/timings.jsonl"},
    "totals": performance,
    "http": http_retries(DATA_DIR),
    "validation_session": {"session": SESSION, "commit": COMMIT, "machine": MACHINE},
    "notes": (f"Colab {RUN_ID}; crawl vào /content/, không phải Drive. "
              "total_cooldown_seconds của merge_benchmarks không áp dụng cho Engine (Engine không ghi "
              "logs/cooldowns.jsonl); thời gian chờ giữa các lượt thử lại nằm trong wall time."),
}
claim_path = SESSION_DIR / "benchmark.claim.json"
claim_path.write_text(json.dumps(claim, ensure_ascii=False, indent=2))
report = validate_crawl(DATA_DIR, claim_path)
(SESSION_DIR / "validator.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
THIS["steps"]["validator"] = {"ok": report["ok"], "findings": report["findings"]}
THIS["performance"] = performance
save_run()
print("validator ok:", report["ok"])
for item in report["findings"]:
    print("-", item["code"], item["message"])
print(json.dumps(performance, indent=2))
{k: v for k, v in summary.items() if v is not None}

In [ ]:
# Bộ số liệu cho báo cáo M1: chép nguyên thư mục report/ vào data/m1/ của repo báo cáo.
LABEL = RUN.get("member") or "explicit"
REPORT_DIR = SESSION_DIR / "report"
bench_dir = REPORT_DIR / "benchmark" / LABEL
bench_dir.mkdir(parents=True, exist_ok=True)
testbed = ("Google Colab CPU" if all(s["machine"]["colab"] for s in RUN["sessions"])
           else "; ".join(sorted({s["machine"]["platform"] for s in RUN["sessions"]})))
report_total = dict(total, member=LABEL, run_id=RUN_ID, commit=FIRST_COMMIT, testbed=testbed,
                    finished=bool(RUN.get("crawl_finished")), sessions=len(RUN["sessions"]),
                    papers_requested=len(IDS),
                    discovery_seconds=summary["discovery_seconds_total"],
                    avg_seconds_per_paper=summary["avg_seconds_per_paper"],
                    median_seconds_per_paper=summary["median_seconds_per_paper"],
                    max_seconds_per_paper=summary["max_seconds_per_paper"],
                    http=claim["http"])
(bench_dir / "benchmark_total.json").write_text(json.dumps(report_total, ensure_ascii=False, indent=2))
shutil.copy2(merged_dir / "benchmark_samples_all.csv", bench_dir / "benchmark_samples_all.csv")
snapshot_path = REPORT_DIR / "snapshots" / f"{LABEL}.json"
note = (f"Colab benchmark {RUN_ID}: one cold run of Engine {FIRST_COMMIT[:7]} "
        f"across {len(RUN['sessions'])} session(s), {total['segments']} segment(s)")
THIS["steps"]["report_m1"] = run([sys.executable, "-m", "scripts.report_m1", "--cache", DATA_DIR,
                                  "--output", snapshot_path, "--member", LABEL, "--ids-file", IDS_FILE,
                                  "--benchmark", bench_dir / "benchmark_total.json", "--crawl-note", note],
                                 SESSION_DIR / "report_m1.log")
snap = json.loads(snapshot_path.read_text())
save_run()
st, bm = snap["storage"], snap["benchmark"]
mb = lambda v: None if v is None else round(v / 1e6, 3)
print(json.dumps({"counts": snap["counts"],
                  "before_mb_mean": mb(st["before_extraction_bytes"]),
                  "after_mb_mean": mb(st["complete_publication_retained_bytes"]["mean"]),
                  "size_reduction_fraction": st["size_reduction_fraction"],
                  "refs_per_publication": snap["references"]["per_successful_publication"]["mean"],
                  "wall_hours": round(bm["total_wall_seconds"] / 3600, 2),
                  "mean_seconds_per_publication": bm["mean_seconds_per_publication"],
                  "discovery_seconds": bm["discovery_seconds"],
                  "peak_rss_mb": total["peak_rss_mb"], "avg_rss_mb": total["avg_rss_mb"],
                  "peak_disk_mb": total["peak_disk_mb"], "final_disk_mb": total["final_disk_mb"]},
                 ensure_ascii=False, indent=2))
print("Bộ số liệu báo cáo:", REPORT_DIR)

In [ ]:
# Publication chưa xong (nếu có): trạng thái từng task để chẩn đoán.
import csv
rows = list(csv.DictReader(open(stats_dir / "papers.csv", encoding="utf-8")))
print({s: sum(r["status"] == s for r in rows) for s in sorted({r["status"] for r in rows})})
pending = [r["id"] for r in rows if r["status"] != "done"]
state = task_state(DATA_DIR, set(pending))
{i: {k.split("|", 1)[1]: v[:2] for k, v in state.items() if k.startswith(i + "|")} for i in pending[:20]}

In [ ]:
# Lưu: checkpoint cuối (gồm thống kê và log của phiên này) và các file kết quả nhỏ lên Drive.
# Ô này in TRẠNG THÁI: XONG / CHƯA XONG.
checkpoint()
if SAVE_TO_DRIVE:
    results = Path(DRIVE_DIR, RUN_ID, SESSION)
    results.mkdir(parents=True, exist_ok=False)
    for path in (RUN_DIR / "run.json", merged_dir / "benchmark_total.json", stats_dir / "summary.json",
                 stats_dir / "papers.csv", claim_path, SESSION_DIR / "validator.json"):
        shutil.copy2(path, results / path.name)
    shutil.copytree(REPORT_DIR, results / "report")
    print("Kết quả nhỏ:", results, "| chép results/report/* vào data/m1/ của repo báo cáo")
print('Phiên mới: RESTORE_ARCHIVE = "' + RUN["checkpoints"][-1].get("drive", RUN["checkpoints"][-1].get("local", "")) + '"')
print()
if RUN.get("crawl_finished") and report["ok"]:
    print("TRẠNG THÁI: XONG. Chia sẻ thư mục", Path(DRIVE_DIR, RUN_ID), "theo ô Nộp kết quả.")
    print(f"Tóm tắt gửi nhóm: MEMBER={LABEL} RUN_ID={RUN_ID} commit={FIRST_COMMIT[:7]} "
          f"hoàn chỉnh={snap['counts']['complete']}/{snap['counts']['assigned']} "
          f"wall={round(total['total_wall_seconds'] / 3600, 2)} giờ phiên={len(RUN['sessions'])}")
elif not RUN.get("crawl_finished"):
    print("CHƯA XONG: mở runtime mới, đặt RESTORE_ARCHIVE như dòng trên, giữ nguyên REF, Run all.")
else:
    print("CHƯA ĐẠT: validator báo lỗi (xem ô validator). Báo nhóm trưởng kèm file run.json.")

In [ ]:
# Tùy chọn: upload Publication hoàn chỉnh lên HF (raw/<MEMBER>/, tables/<MEMBER>/). Chỉ đọc cache local.
if UPLOAD_TO_HF:
    assert RUN.get("crawl_finished") and report["ok"], "Chỉ upload sau khi crawl xong và validator đạt"
    from dataclasses import replace
    from src.common.config import get_settings
    from src.crawl.api import run_crawl
    os.environ["MEMBER_NAME"] = RUN["member"]
    settings = replace(get_settings(), data_dir=DATA_DIR, member=RUN["member"])
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    uploaded = run_crawl(IDS, settings=settings, upload=True, upload_only=True, batch_size=250)
    THIS["steps"]["upload"] = {"counts": uploaded["counts"]}
    save_run()
    print(uploaded["counts"])   # not_ready > 0 là bình thường khi còn Publication chưa hoàn chỉnh

In [ ]:
# Tải run.json về máy để dán vào báo cáo (dữ liệu crawl nằm trong checkpoint trên Drive).
if IN_COLAB:
    from google.colab import files
    files.download(str(RUN_DIR / "run.json"))

## 📤 Nộp kết quả (khi ô Lưu in `TRẠNG THÁI: XONG`)

1. Mở Google Drive → `MyDrive/arxiv-m1-benchmark/` → thư mục **`<RUN_ID>`** (ô Lưu in ra,
   dạng `2026-11-13T020000Z_abc1234`). Bên trong có một thư mục cho mỗi phiên; phiên cuối có
   `report/`, `run.json`, `summary.json`, `papers.csv`, `validator.json`, `benchmark_total.json`.
2. Chuột phải thư mục `<RUN_ID>` → *Share* → thêm **Bảo** với quyền **Viewer**
   (*Người xem*), hoặc *Copy link* với chế độ "Anyone with the link: Viewer".
3. Gửi vào nhóm chat: link thư mục + dòng **Tóm tắt gửi nhóm** mà ô Lưu in ra.
4. **Giữ** file `<RUN_ID>__checkpoint.tar.gz` trên Drive (chứa toàn bộ dữ liệu crawl), không
   cần chia sẻ trừ khi nhóm trưởng yêu cầu; không xóa trước khi nộp báo cáo M1.

Nhóm trưởng chép `report/` của từng Range vào `data/m1/` của repo báo cáo; mọi bảng và hình
của trang *Kết quả thu thập* và *Hiệu năng* sinh lại từ đó.

## 🛠️ Gặp lỗi

| Thông báo | Làm gì |
|---|---|
| `Thêm CONTACT_EMAIL trong Colab Secrets` | Bước 0.3: thêm secret và bật *Notebook access*, chạy lại |
| `REF chưa có ...` | Bạn đang mở notebook/REF cũ: mở lại từ link `blob/m1-bench-v2/`, `REF = "m1-bench-v2"` |
| `pytest thất bại` | Gửi file `pytest.log` (đường dẫn in trong ô) cho nhóm trưởng; không chạy tiếp |
| `commit phiên này ... khác commit phiên đầu` | Phiên tiếp theo phải giữ đúng `REF` của phiên đầu |
| `... đã tồn tại; dùng runtime mới` | *Disconnect and delete runtime* rồi chạy lại |
| Colab ngắt kết nối / tab bị đóng | Không sao: mở runtime mới, `RESTORE_ARCHIVE` = checkpoint gần nhất trên Drive, *Run all* |
| Dòng log có `429` | Bình thường: arXiv yêu cầu chờ, Engine tự chờ rồi thử lại; số lần được ghi vào kết quả |
| `Crawler dừng bất thường` | Gửi dòng lỗi + file `crawl-run_NNN.log` cho nhóm trưởng |
| Drive đầy | Checkpoint một Range khoảng 0,5–2 GB (giữ 2 bản: hiện tại và `.prev`); dọn Drive rồi chạy lại ô crawl |